# Modeling Approach

Based on the EDA, bike rental demand appears to be influenced by temporal, seasonal and weather-related factors. Therefore, several regression models are trained and compared.

The selected models cover different levels of complexity:

1. Dummy Regressor as baseline
2. Linear Regression as interpretable benchmark
3. Ridge Regression to address multicollinearity and regularization
4. Random Forest Regressor to capture non-linear relationships
5. HistGradientBoostingRegressor to model complex interactions more effectively

All models are evaluated using MAE, RMSE and R².

## General Notes
### Excluded Columns

Before training the models, some columns are excluded from the feature set. This is necessary to avoid misleading results and to ensure that the models learn meaningful relationships from the data.

#### Target Variable

The column `cnt` is excluded from the input features because it is the target variable that should be predicted.

Including `cnt` as an input feature would make the prediction task invalid, since the model would already have access to the value it is supposed to predict.

---

#### Leakage Features

The columns `casual` and `registered` are also excluded from the feature set.

These two variables represent the number of casual and registered users, and together they directly form the target variable:

`cnt = casual + registered`

If these columns were included, the model would receive direct information about the target variable. This would lead to data leakage and artificially high model performance. In practice, such a model would not be useful because these values would usually not be known before predicting total demand.

---

#### Identifier Column

The column `instant` is removed because it only represents a sequential identifier for each observation.

It does not contain meaningful information about bike rental demand and could introduce artificial patterns into the model.

---

#### Date Column

The column `dteday` is not used directly as a model feature because it is stored as a date value and cannot be interpreted meaningfully by most machine learning models in its raw form.

The raw date column is excluded because relevant temporal information is already available through variables such as `yr`, `mnth`, `weekday`, and `hr`. Additional date-based features can be derived separately if needed.

---

#### Summary

The following columns are excluded before model training:

- `cnt`: target variable
- `casual`: leakage feature
- `registered`: leakage feature
- `instant`: non-informative identifier
- `dteday`: raw date column, already represented by derived temporal features

This ensures that the models are trained only on features that would realistically be available before making a prediction.

## Feature Engineering
In this section, the original dataset is transformed into a model-ready feature set.

The goal of feature engineering is to create additional variables that better represent the patterns observed during the exploratory data analysis. In particular, the EDA indicated that bike rental demand is strongly influenced by temporal structures, commuting behavior, seasonal effects, and weather-related conditions.

Therefore, new features are derived from existing variables, including date-based features, cyclical time encodings, interaction features, rush-hour indicators, daytime categories, and weekend indicators.

At the same time, invalid or non-informative columns are removed from the feature set. This includes the target variable, leakage features, identifiers, and the raw date column.

The resulting feature matrix serves as the basis for the following preprocessing and model training steps.

### Imports

In [8]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv("./hour.csv")

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


### Create Working Copy

A copy of the original dataset is created to avoid modifying the raw data directly.

In [10]:
df_fe = df.copy() # fe for feature engineering

df_fe.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


### Convert Date Column

In [25]:
df_fe["dteday"] = pd.to_datetime(df_fe["dteday"])

df_fe = df_fe.sort_values(["dteday", "hr"]).reset_index(drop=True)

df_fe["dteday"].head()

0   2011-01-01
1   2011-01-01
2   2011-01-01
3   2011-01-01
4   2011-01-01
Name: dteday, dtype: datetime64[ns]

### Create Additional Date-Based Features

Additional date-based features are extracted from `dteday`.

These features may help models capture calendar-related effects more explicitly.

In [13]:
df_fe["day"] = df_fe["dteday"].dt.day
df_fe["dayofyear"] = df_fe["dteday"].dt.dayofyear

# Custom week of year based on dayofyear
# This avoids ISO calendar behavior where early January dates can belong to the last week of the previous year.
df_fe["weekofyear"] = ((df_fe["dayofyear"] - 1) // 7 + 1).astype(int)

df_fe[["dteday", "day", "dayofyear", "weekofyear"]].head()

,dteday,day,dayofyear,weekofyear
0,2011-01-01,1,1,1
1,2011-01-01,1,1,1
2,2011-01-01,1,1,1
3,2011-01-01,1,1,1
4,2011-01-01,1,1,1


### Create Cyclical Time Features

Some temporal variables are cyclical.  
For example, hour 23 and hour 0 are close in reality, although they are numerically far apart.

To represent this structure better, sine and cosine transformations are created for:

- hour of day
- weekday
- month

These features are especially useful for linear models.

In [14]:
# Hour: 0–23
df_fe["hr_sin"] = np.sin(2 * np.pi * df_fe["hr"] / 24)
df_fe["hr_cos"] = np.cos(2 * np.pi * df_fe["hr"] / 24)

# Weekday: 0–6
df_fe["weekday_sin"] = np.sin(2 * np.pi * df_fe["weekday"] / 7)
df_fe["weekday_cos"] = np.cos(2 * np.pi * df_fe["weekday"] / 7)

# Month: 1–12
df_fe["mnth_sin"] = np.sin(2 * np.pi * df_fe["mnth"] / 12)
df_fe["mnth_cos"] = np.cos(2 * np.pi * df_fe["mnth"] / 12)

df_fe[
    [
        "hr", "hr_sin", "hr_cos",
        "weekday", "weekday_sin", "weekday_cos",
        "mnth", "mnth_sin", "mnth_cos"
    ]
].head()

,hr,hr_sin,hr_cos,weekday,weekday_sin,weekday_cos,mnth,mnth_sin,mnth_cos
0,0,0.000000,1.000000,6,-0.781831,0.62349,1,0.5,0.866025
1,1,0.258819,0.965926,6,-0.781831,0.62349,1,0.5,0.866025
2,2,0.500000,0.866025,6,-0.781831,0.62349,1,0.5,0.866025
3,3,0.707107,0.707107,6,-0.781831,0.62349,1,0.5,0.866025
4,4,0.866025,0.500000,6,-0.781831,0.62349,1,0.5,0.866025


### Create Interaction Features

The EDA showed that the effect of some variables may depend on other variables.

For example, the impact of temperature may differ between seasons, and environmental conditions such as humidity or wind may interact with weather categories.

Therefore, selected interaction features are created.

In [15]:
df_fe["temp_x_season"] = df_fe["temp"] * df_fe["season"]
df_fe["temp_x_workingday"] = df_fe["temp"] * df_fe["workingday"]
df_fe["hum_x_weathersit"] = df_fe["hum"] * df_fe["weathersit"]
df_fe["windspeed_x_weathersit"] = df_fe["windspeed"] * df_fe["weathersit"]

df_fe[
    [
        "temp", "season", "temp_x_season",
        "workingday", "temp_x_workingday",
        "hum", "weathersit", "hum_x_weathersit",
        "windspeed", "windspeed_x_weathersit"
    ]
].head()

,temp,season,temp_x_season,workingday,temp_x_workingday,hum,weathersit,hum_x_weathersit,windspeed,windspeed_x_weathersit
0,0.24,1,0.24,0,0.0,0.81,1,0.81,0.0,0.0
1,0.22,1,0.22,0,0.0,0.80,1,0.80,0.0,0.0
2,0.22,1,0.22,0,0.0,0.80,1,0.80,0.0,0.0
3,0.24,1,0.24,0,0.0,0.75,1,0.75,0.0,0.0
4,0.24,1,0.24,0,0.0,0.75,1,0.75,0.0,0.0


### Create Rush Hour Feature

The EDA showed clear demand peaks during typical commuting hours.

A binary feature is created to indicate whether an observation belongs to a rush hour period.

In [16]:
df_fe["rush_hour"] = df_fe["hr"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

df_fe[["hr", "rush_hour"]].head(12)

,hr,rush_hour
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,1
8,8,1
9,9,1


### Create Daytime Category

The hour of the day is grouped into broader time periods.

This feature may help models capture general daily usage phases such as morning, afternoon, evening, and night.

In [17]:
def get_daytime(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"


df_fe["daytime"] = df_fe["hr"].apply(get_daytime)

df_fe[["hr", "daytime"]].head(24)

,hr,daytime
0,0,night
1,1,night
2,2,night
3,3,night
4,4,night
5,5,morning
6,6,morning
7,7,morning
8,8,morning
9,9,morning


### Create Weekend Feature

A binary weekend feature is created based on the weekday variable.

This provides an additional distinction between regular weekdays and weekends.

In [18]:
df_fe["is_weekend"] = df_fe["weekday"].isin([0, 6]).astype(int)

df_fe[["weekday", "is_weekend"]].drop_duplicates().sort_values("weekday")

,weekday,is_weekend
24,0,1
47,1,0
69,2,0
92,3,0
115,4,0
138,5,0
0,6,1


## Lag Features

Since the dataset contains hourly observations, previous demand values can provide useful information for predicting current bike rental demand.

Lag features are created based on the target variable `cnt`. They represent demand in previous time steps, such as the previous hour, the same hour on the previous day, or the same hour in the previous week.

Only past values are used to avoid target leakage.

In [26]:
# Lag features based on previous demand
df_fe["cnt_lag_1"] = df_fe["cnt"].shift(1)
df_fe["cnt_lag_2"] = df_fe["cnt"].shift(2)
df_fe["cnt_lag_24"] = df_fe["cnt"].shift(24)
df_fe["cnt_lag_48"] = df_fe["cnt"].shift(48)
df_fe["cnt_lag_168"] = df_fe["cnt"].shift(168)

df_fe[
    [
        "dteday", "hr", "cnt",
        "cnt_lag_1", "cnt_lag_2",
        "cnt_lag_24", "cnt_lag_48", "cnt_lag_168"
    ]
].head(30)

,dteday,hr,cnt,cnt_lag_1,cnt_lag_2,cnt_lag_24,cnt_lag_48,cnt_lag_168
0,2011-01-01,0,16,NaN,NaN,NaN,NaN,NaN
1,2011-01-01,1,40,16.0,NaN,NaN,NaN,NaN
2,2011-01-01,2,32,40.0,16.0,NaN,NaN,NaN
3,2011-01-01,3,13,32.0,40.0,NaN,NaN,NaN
4,2011-01-01,4,1,13.0,32.0,NaN,NaN,NaN
5,2011-01-01,5,1,1.0,13.0,NaN,NaN,NaN
6,2011-01-01,6,2,1.0,1.0,NaN,NaN,NaN
7,2011-01-01,7,3,2.0,1.0,NaN,NaN,NaN
8,2011-01-01,8,8,3.0,2.0,NaN,NaN,NaN
9,2011-01-01,9,14,8.0,3.0,NaN,NaN,NaN


### Rolling Window Features

Rolling window features summarize recent demand patterns over a defined time window.

To avoid target leakage, the target variable is shifted before applying rolling calculations. This ensures that the current value of `cnt` is not included in the rolling statistics used to predict itself.

In [27]:
# Rolling features based only on past demand
df_fe["cnt_roll_3_mean"] = df_fe["cnt"].shift(1).rolling(window=3).mean()
df_fe["cnt_roll_6_mean"] = df_fe["cnt"].shift(1).rolling(window=6).mean()
df_fe["cnt_roll_24_mean"] = df_fe["cnt"].shift(1).rolling(window=24).mean()
df_fe["cnt_roll_24_std"] = df_fe["cnt"].shift(1).rolling(window=24).std()

df_fe[
    [
        "dteday", "hr", "cnt",
        "cnt_roll_3_mean",
        "cnt_roll_6_mean",
        "cnt_roll_24_mean",
        "cnt_roll_24_std"
    ]
].head(30)

,dteday,hr,cnt,cnt_roll_3_mean,cnt_roll_6_mean,cnt_roll_24_mean,cnt_roll_24_std
0,2011-01-01,0,16,NaN,NaN,NaN,NaN
1,2011-01-01,1,40,NaN,NaN,NaN,NaN
2,2011-01-01,2,32,NaN,NaN,NaN,NaN
3,2011-01-01,3,13,29.333333,NaN,NaN,NaN
4,2011-01-01,4,1,28.333333,NaN,NaN,NaN
5,2011-01-01,5,1,15.333333,NaN,NaN,NaN
6,2011-01-01,6,2,5.000000,17.166667,NaN,NaN
7,2011-01-01,7,3,1.333333,14.833333,NaN,NaN
8,2011-01-01,8,8,2.000000,8.666667,NaN,NaN
9,2011-01-01,9,14,4.333333,4.666667,NaN,NaN


### Remove Rows with Missing Lag and Rolling Features

Lag and rolling features create missing values at the beginning of the dataset because no sufficient historical information is available for the first observations.

These rows are removed before model training.

In [28]:
df_fe = df_fe.dropna().reset_index(drop=True)

df_fe.isna().sum().sum()

np.int64(0)

### Define Target Variable

The target variable is `cnt`, representing the total number of bike rentals per hour.

In [31]:
y = df_fe["cnt"]

y.head()

0     9
1    15
2    20
3    61
4    62
Name: cnt, dtype: int64

### Remove Excluded Columns

Several columns are excluded before model training:

- `cnt`: target variable
- `casual`: leakage feature
- `registered`: leakage feature
- `instant`: non-informative identifier
- `dteday`: raw date column

This prevents target leakage and ensures that only meaningful predictor variables are used.

In [32]:
columns_to_drop = [
    "cnt",
    "casual",
    "registered",
    "instant",
    "dteday"
]

X = df_fe.drop(columns=columns_to_drop)

X.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,...,is_weekend,cnt_lag_1,cnt_lag_2,cnt_lag_24,cnt_lag_48,cnt_lag_168,cnt_roll_3_mean,cnt_roll_6_mean,cnt_roll_24_mean,cnt_roll_24_std
0,1,0,1,7,0,6,0,2,0.16,0.1818,...,1,2.0,5.0,84.0,36.0,16.0,2.666667,7.833333,63.208333,55.844488
1,1,0,1,8,0,6,0,3,0.16,0.1818,...,1,9.0,2.0,210.0,95.0,40.0,5.333333,6.666667,60.083333,56.721989
2,1,0,1,9,0,6,0,3,0.16,0.1818,...,1,15.0,9.0,134.0,219.0,32.0,8.666667,6.500000,51.958333,47.536237
3,1,0,1,10,0,6,0,2,0.18,0.1970,...,1,20.0,15.0,63.0,122.0,13.0,14.666667,8.666667,47.208333,44.585998
4,1,0,1,11,0,6,0,2,0.20,0.1818,...,1,61.0,20.0,67.0,45.0,1.0,32.000000,18.666667,47.125000,44.557059


### Check Final Feature Matrix

The final feature matrix is checked before preprocessing and model training.

In [22]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()
X.dtypes

Feature matrix shape: (17379, 28)
Target shape: (17379,)


season                      int64
yr                          int64
mnth                        int64
hr                          int64
holiday                     int64
weekday                     int64
workingday                  int64
weathersit                  int64
temp                      float64
atemp                     float64
hum                       float64
windspeed                 float64
day                         int32
dayofyear                   int32
weekofyear                  int64
hr_sin                    float64
hr_cos                    float64
weekday_sin               float64
weekday_cos               float64
mnth_sin                  float64
mnth_cos                  float64
temp_x_season             float64
temp_x_workingday         float64
hum_x_weathersit          float64
windspeed_x_weathersit    float64
rush_hour                   int64
daytime                    object
is_weekend                  int64
dtype: object

### Identify Categorical, Binary and Numerical Features

For later preprocessing, the features are separated into categorical, binary, and numerical variables.

This is necessary because different feature types require different preprocessing steps.

In [33]:
categorical_features = [
    "season",
    "mnth",
    "hr",
    "weekday",
    "weathersit",
    "daytime"
]

binary_features = [
    "yr",
    "holiday",
    "workingday",
    "rush_hour",
    "is_weekend"
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features + binary_features
]

print("Categorical features:")
print(categorical_features)

print("\nBinary features:")
print(binary_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['season', 'mnth', 'hr', 'weekday', 'weathersit', 'daytime']

Binary features:
['yr', 'holiday', 'workingday', 'rush_hour', 'is_weekend']

Numerical features:
['temp', 'atemp', 'hum', 'windspeed', 'day', 'dayofyear', 'weekofyear', 'hr_sin', 'hr_cos', 'weekday_sin', 'weekday_cos', 'mnth_sin', 'mnth_cos', 'temp_x_season', 'temp_x_workingday', 'hum_x_weathersit', 'windspeed_x_weathersit', 'cnt_lag_1', 'cnt_lag_2', 'cnt_lag_24', 'cnt_lag_48', 'cnt_lag_168', 'cnt_roll_3_mean', 'cnt_roll_6_mean', 'cnt_roll_24_mean', 'cnt_roll_24_std']


### Validate Feature Groups

Before continuing, the feature groups are checked to ensure that every feature is assigned to exactly one group.

In [24]:
all_grouped_features = categorical_features + binary_features + numerical_features

print("Number of columns in X:", len(X.columns))
print("Number of grouped features:", len(all_grouped_features))

missing_features = set(X.columns) - set(all_grouped_features)
extra_features = set(all_grouped_features) - set(X.columns)
duplicate_count = len(all_grouped_features) - len(set(all_grouped_features))

print("Missing features:", missing_features)
print("Extra features:", extra_features)
print("Number of duplicate assignments:", duplicate_count)

Number of columns in X: 28
Number of grouped features: 28
Missing features: set()
Extra features: set()
Number of duplicate assignments: 0


### Feature Engineering Summary

The feature engineering process created additional variables based on temporal and environmental patterns observed during the EDA.

The following feature types were added:

- Date-based features such as day, day of year, and calendar week
- Cyclical encodings for hour, weekday, and month
- Interaction features between weather, season, and working day variables
- A rush hour indicator based on commuting peaks
- A daytime category
- A weekend indicator

Leakage features and non-informative columns were removed before modeling.

The resulting feature matrix can now be used for preprocessing pipelines and model training.

## Preprocessing
In this section, the prepared feature matrix is split into training and test data.  
Afterwards, separate preprocessing strategies are defined for different model types.

The preprocessing is designed to avoid data leakage by fitting transformations only on the training data.

### Check Feature Matrix and Target Variable

Before splitting the data, the feature matrix `X` and target variable `y` are checked.

In [34]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Feature matrix shape: (17211, 37)
Target shape: (17211,)


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,...,is_weekend,cnt_lag_1,cnt_lag_2,cnt_lag_24,cnt_lag_48,cnt_lag_168,cnt_roll_3_mean,cnt_roll_6_mean,cnt_roll_24_mean,cnt_roll_24_std
0,1,0,1,7,0,6,0,2,0.16,0.1818,...,1,2.0,5.0,84.0,36.0,16.0,2.666667,7.833333,63.208333,55.844488
1,1,0,1,8,0,6,0,3,0.16,0.1818,...,1,9.0,2.0,210.0,95.0,40.0,5.333333,6.666667,60.083333,56.721989
2,1,0,1,9,0,6,0,3,0.16,0.1818,...,1,15.0,9.0,134.0,219.0,32.0,8.666667,6.500000,51.958333,47.536237
3,1,0,1,10,0,6,0,2,0.18,0.1970,...,1,20.0,15.0,63.0,122.0,13.0,14.666667,8.666667,47.208333,44.585998
4,1,0,1,11,0,6,0,2,0.20,0.1818,...,1,61.0,20.0,67.0,45.0,1.0,32.000000,18.666667,47.125000,44.557059


0     9
1    15
2    20
3    61
4    62
Name: cnt, dtype: int64

### Define Feature Groups

The features are grouped into categorical, binary, and numerical variables.

This is necessary because different feature types require different preprocessing steps:

- categorical features are one-hot encoded
- binary features are passed through unchanged
- numerical features are scaled for linear models but not for tree-based models

In [35]:
categorical_features = [
    "season",
    "mnth",
    "hr",
    "weekday",
    "weathersit",
    "daytime"
]

binary_features = [
    "yr",
    "holiday",
    "workingday",
    "rush_hour",
    "is_weekend"
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features + binary_features
]

print("Categorical features:")
print(categorical_features)

print("\nBinary features:")
print(binary_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['season', 'mnth', 'hr', 'weekday', 'weathersit', 'daytime']

Binary features:
['yr', 'holiday', 'workingday', 'rush_hour', 'is_weekend']

Numerical features:
['temp', 'atemp', 'hum', 'windspeed', 'day', 'dayofyear', 'weekofyear', 'hr_sin', 'hr_cos', 'weekday_sin', 'weekday_cos', 'mnth_sin', 'mnth_cos', 'temp_x_season', 'temp_x_workingday', 'hum_x_weathersit', 'windspeed_x_weathersit', 'cnt_lag_1', 'cnt_lag_2', 'cnt_lag_24', 'cnt_lag_48', 'cnt_lag_168', 'cnt_roll_3_mean', 'cnt_roll_6_mean', 'cnt_roll_24_mean', 'cnt_roll_24_std']


In [36]:
all_grouped_features = categorical_features + binary_features + numerical_features

missing_features = set(X.columns) - set(all_grouped_features)
extra_features = set(all_grouped_features) - set(X.columns)
duplicate_count = len(all_grouped_features) - len(set(all_grouped_features))

print("Number of columns in X:", len(X.columns))
print("Number of grouped features:", len(all_grouped_features))

print("\nMissing features:", missing_features)
print("Extra features:", extra_features)
print("Number of duplicate assignments:", duplicate_count)

Number of columns in X: 37
Number of grouped features: 37

Missing features: set()
Extra features: set()
Number of duplicate assignments: 0


### Chronological Train-Test Split

A chronological train-test split is used because the observations are time-dependent.

The first 80% of the observations are used for training, while the last 20% are kept as a final holdout test set.

This reflects a realistic forecasting scenario where future demand is predicted based on past observations.

In [37]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (13768, 37)
X_test shape: (3443, 37)
y_train shape: (13768,)
y_test shape: (3443,)


### Check Train-Test Split

The split is checked to make sure that the training and test sets are separated correctly.

In [38]:
print("Training set:")
print("Start index:", X_train.index.min())
print("End index:", X_train.index.max())

print("\nTest set:")
print("Start index:", X_test.index.min())
print("End index:", X_test.index.max())

Training set:
Start index: 0
End index: 13767

Test set:
Start index: 13768
End index: 17210


### Time-Series Cross-Validation

Instead of standard k-fold cross-validation, `TimeSeriesSplit` is used.

This ensures that, in each validation fold, the training data always comes before the validation data. This avoids leakage from future observations into the training process.

The final test set remains untouched and is only used for the final evaluation.

In [39]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    print(f"Fold {fold}")
    print("Train indices:", train_idx[0], "to", train_idx[-1])
    print("Validation indices:", val_idx[0], "to", val_idx[-1])
    print("Train size:", len(train_idx))
    print("Validation size:", len(val_idx))
    print()

Fold 1
Train indices: 0 to 2297
Validation indices: 2298 to 4591
Train size: 2298
Validation size: 2294

Fold 2
Train indices: 0 to 4591
Validation indices: 4592 to 6885
Train size: 4592
Validation size: 2294

Fold 3
Train indices: 0 to 6885
Validation indices: 6886 to 9179
Train size: 6886
Validation size: 2294

Fold 4
Train indices: 0 to 9179
Validation indices: 9180 to 11473
Train size: 9180
Validation size: 2294

Fold 5
Train indices: 0 to 11473
Validation indices: 11474 to 13767
Train size: 11474
Validation size: 2294



In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Preprocessor for Linear Models

Linear models require numerical variables to be on a comparable scale and cannot directly process categorical variables.

Therefore, the preprocessing for linear models includes:

- one-hot encoding for categorical variables
- standard scaling for numerical variables
- passing binary variables through unchanged

This preprocessor will later be used for Linear Regression and Ridge Regression.

In [41]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ],
    remainder="drop"
)

linear_preprocessor

ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['season', 'mnth', 'hr', 'weekday',
                                  'weathersit', 'daytime']),
                                ('numerical', StandardScaler(),
                                 ['temp', 'atemp', 'hum', 'windspeed', 'day',
                                  'dayofyear', 'weekofyear', 'hr_sin', 'hr_cos',
                                  'weekday_sin', 'weekday_cos', 'mnth_sin',
                                  'mnth_cos', 'temp_x_season',
                                  'temp_x_workingday', 'hum_x_weathersit',
                                  'windspeed_x_weathersit', 'cnt_lag_1',
                                  'cnt_lag_2', 'cnt_lag_24', 'cnt_lag_48',
                                  'cnt_lag_168', 'cnt_roll_3_mean',
                                  'cnt_roll_6_mean', 'cnt_roll_24_mean',
                                  'cnt_roll_24_std']),
                                ('binary', 'passthrough',
                                 ['yr', 'holiday', 'workingday', 'rush_hour',
                                  'is_weekend'])])

### Preprocessor for Tree-Based Models

Tree-based models do not require feature scaling because they split features based on thresholds.

However, categorical variables are still one-hot encoded to avoid treating category labels as meaningful numerical orders.

Therefore, the preprocessing for tree-based models includes:

- one-hot encoding for categorical variables
- passing numerical variables through unchanged
- passing binary variables through unchanged

This preprocessor will later be used for Random Forest and HistGradientBoostingRegressor.

In [42]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ],
    remainder="drop"
)

tree_preprocessor

ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['season', 'mnth', 'hr', 'weekday',
                                  'weathersit', 'daytime']),
                                ('numerical', 'passthrough',
                                 ['temp', 'atemp', 'hum', 'windspeed', 'day',
                                  'dayofyear', 'weekofyear', 'hr_sin', 'hr_cos',
                                  'weekday_sin', 'weekday_cos', 'mnth_sin',
                                  'mnth_cos', 'temp_x_season',
                                  'temp_x_workingday', 'hum_x_weathersit',
                                  'windspeed_x_weathersit', 'cnt_lag_1',
                                  'cnt_lag_2', 'cnt_lag_24', 'cnt_lag_48',
                                  'cnt_lag_168', 'cnt_roll_3_mean',
                                  'cnt_roll_6_mean', 'cnt_roll_24_mean',
                                  'cnt_roll_24_std']),
                                ('binary', 'passthrough',
                                 ['yr', 'holiday', 'workingday', 'rush_hour',
                                  'is_weekend'])])

### Test Preprocessing Output

The preprocessors are applied once to the training data to verify that the transformations work correctly.

In the actual modeling step, the preprocessors will be used inside full model pipelines.

In [43]:
X_train_linear_processed = linear_preprocessor.fit_transform(X_train)
X_train_tree_processed = tree_preprocessor.fit_transform(X_train)

print("Linear preprocessing output shape:", X_train_linear_processed.shape)
print("Tree preprocessing output shape:", X_train_tree_processed.shape)

Linear preprocessing output shape: (13768, 80)
Tree preprocessing output shape: (13768, 80)


### Extract Transformed Feature Names

After one-hot encoding, categorical variables are expanded into multiple columns.

Extracting the transformed feature names is useful for later model interpretation, especially for coefficients and feature importance.

In [44]:
linear_feature_names = linear_preprocessor.get_feature_names_out()
tree_feature_names = tree_preprocessor.get_feature_names_out()

print("Number of linear model features:", len(linear_feature_names))
print("Number of tree-based model features:", len(tree_feature_names))

linear_feature_names[:20]

Number of linear model features: 80
Number of tree-based model features: 80


array(['categorical__season_2', 'categorical__season_3',
       'categorical__season_4', 'categorical__mnth_2',
       'categorical__mnth_3', 'categorical__mnth_4',
       'categorical__mnth_5', 'categorical__mnth_6',
       'categorical__mnth_7', 'categorical__mnth_8',
       'categorical__mnth_9', 'categorical__mnth_10',
       'categorical__mnth_11', 'categorical__mnth_12',
       'categorical__hr_1', 'categorical__hr_2', 'categorical__hr_3',
       'categorical__hr_4', 'categorical__hr_5', 'categorical__hr_6'],
      dtype=object)

### Preprocessing Summary

The preprocessing step prepared the data for model training while preserving the temporal structure of the dataset.

A chronological holdout split was used to create a final test set.  
Additionally, `TimeSeriesSplit` was defined for cross-validation on the training data.

Two preprocessing strategies were created:

1. **Linear model preprocessing**
   - one-hot encoding for categorical features
   - scaling of numerical features
   - binary features passed through unchanged

2. **Tree-based model preprocessing**
   - one-hot encoding for categorical features
   - numerical and binary features passed through unchanged

These preprocessing objects can now be combined with different regression models in scikit-learn pipelines.

## Model Training

## Evaluation